In [2]:
# PATHS
dataset = r"F:\SNAP-5IMG-10C"

In [ ]:
import os
def walk_through_dir(dir_path):
  for dirpath, dirnames, filenames in os.walk(dir_path):
    print(f"There are {len(dirnames)} directories and {len(filenames)} images in '{dirpath}'.")

walk_through_dir(dataset)

In [9]:
import os
import matplotlib.pyplot as plt

def count_images_in_directories(base_path):
    directory_counts = {}
    directory_names = []
    
    # Walk through the directory to get the subdirectories and file counts
    for dirpath, dirnames, filenames in os.walk(base_path):
        if dirpath == base_path:
            # Skip the root directory
            continue
        directory_name = os.path.basename(dirpath)
        image_count = len(filenames)
        directory_counts[directory_name] = image_count
        
        # Collect directory names for ordering
        directory_names.append(directory_name)
    
    return directory_counts, sorted(directory_names)

# Get the image counts and sorted directory names
image_counts, ordered_directories = count_images_in_directories(dataset)

# Create a map from directory name to index
directory_map = {name: idx for idx, name in enumerate(ordered_directories)}

# Print the directory map with their corresponding image counts
for key, value in image_counts.items():
    class_number = directory_map.get(key, None)
    if class_number is not None:
        print(f'{class_number}: {key} - {value} images')

0: Asis - 10 images
1: Ilang Ilang - 10 images
2: Kamagong - 10 images
3: Mahogany - 10 images
4: Narra - 10 images


In [ ]:
import os
import torch
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from PIL import Image

# Set the model and device
model_id = r"E:\Snapfolia - CS\grounding-dino-tiny"
device = "cuda" if torch.cuda.is_available() else "cpu"

# Initialize processor and model
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForZeroShotObjectDetection.from_pretrained(model_id).to(device)

# Set the root directory containing the classes and images
root_dir = "F:\SNAP-5IMG-10C"

# Text to detect (leaves in this case)
text = "a leaf. a leaves."

# Create the output directory for the extracted data
output_root_dir = "F:\SNAP-5IM-10C-EXTRACTED"
os.makedirs(output_root_dir, exist_ok=True)

# Initialize class index mapping
class_names = os.listdir(root_dir)
class_name_to_index = {class_name: idx for idx, class_name in enumerate(class_names)}

# Iterate over all classes in the root directory
for class_name, class_idx in class_name_to_index.items():
    class_folder = os.path.join(root_dir, class_name)
    
    if os.path.isdir(class_folder):
        print()
        print("--------------------------------------------------------")
        print(f"Processing class: {class_name} (Class {class_idx})")
        
        # Create a directory to save results for each class (use lowercase for class names)
        class_output_dir = os.path.join(output_root_dir, f"class{class_idx}")
        os.makedirs(class_output_dir, exist_ok=True)
        
        # Create subdirectories for images and labels
        images_dir = os.path.join(class_output_dir, "images")
        labels_dir = os.path.join(class_output_dir, "labels")
        os.makedirs(images_dir, exist_ok=True)
        os.makedirs(labels_dir, exist_ok=True)
        
        # Iterate over all images in the class folder
        for image_name in os.listdir(class_folder):
            image_path = os.path.join(class_folder, image_name)
            
            if image_name.endswith(('.jpg', '.jpeg', '.png')):
                print(f"->Processing image: {image_name}")
                
                # Open the image
                image = Image.open(image_path)

                # Prepare the inputs for the model
                inputs = processor(images=image, text=text, return_tensors="pt").to(device)
                
                # Perform inference
                with torch.no_grad():
                    outputs = model(**inputs)

                # Post-process the results
                results = processor.post_process_grounded_object_detection(
                    outputs,
                    inputs.input_ids,
                    box_threshold=0.3,
                    text_threshold=0.3,
                    target_sizes=[image.size[::-1]]
                )

                # Create the output file for labels
                output_file = os.path.join(labels_dir, f"{os.path.splitext(image_name)[0]}.txt")

                # Check if any detections were made
                if len(results) > 0:
                    # Get the first (and likely only) result
                    pred_boxes = results[0]["boxes"]
                    pred_labels = results[0]["labels"]
                    pred_scores = results[0]["scores"]
                    
                    with open(output_file, 'w') as f:
                        # Write coordinates to file in the required format: class_idx x_center y_center width height
                        for i, (box, label, score) in enumerate(zip(pred_boxes, pred_labels, pred_scores), 1):
                            # Convert box coordinates to integers
                            box = [int(b) for b in box]

                            # Write coordinates to file
                            x_center = (box[0] + box[2]) / 2
                            y_center = (box[1] + box[3]) / 2
                            width = box[2] - box[0]
                            height = box[3] - box[1]
                            f.write(f"{class_idx} {x_center} {y_center} {width} {height}\n")

                            # Print detection details
                            # print(f"Leaf {i}: Score = {score:.2f}, Box = {box}")
                else:
                    print(f"No objects detected in {image_name}.")

                # Save the image (without bounding boxes)
                output_image_path = os.path.join(images_dir, f"{os.path.splitext(image_name)[0]}.png")
                image.save(output_image_path)